# Multi-Task Learning

This notebook implements a multi-task learning approach to the
classification task. A single shared transformer encoder is trained
jointly on the main objective and a set of auxiliary tasks, with the goal
of learning richer and more robust representations of the input.

The auxiliary signals and the loss design encourage the model to capture
relationships between classes, complementing the main classification head
used to produce the final prediction.

In [ ]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pandas as pd
from collections import Counter
import transformers
from transformers import AutoTokenizer, AutoModel
from datasets import load_from_disk
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "/content/drive/MyDrive/{hide}"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
transformers.set_seed(42)

Input formatting

In [ ]:
# Concatenate seeker turns

def format_dialogue(dialogue):
  seeker_turns = [t["text"] for t in dialogue if t["speaker"] == "seeker"]
  return " ".join(seeker_turns)


In [ ]:
# Calculate auxiliary labels

def get_aux_labels(label):
    if label == 8:
        return -1, -1, -1, -1
    return (
        int(label != 0),  # has_defense
        int(label in [2, 4]),  # is_image_distorting
        int(label in [5, 6]),  # is_cognitive
        int(label == 7),  # is_mature
    )


In [ ]:
# 9x9 distance matrix based on DMRS theory

PSYCH_DIST = [
    # 0  1  2  3  4  5  6  7  8
    [0, 3, 3, 3, 3, 3, 3, 3, 1],  # 0
    [3, 0, 2, 2, 2, 3, 3, 3, 1],  # 1
    [3, 2, 0, 2, 1, 2, 2, 3, 1],  # 2
    [3, 2, 2, 0, 2, 2, 3, 3, 1],  # 3
    [3, 2, 1, 2, 0, 2, 2, 3, 1],  # 4
    [3, 3, 2, 2, 2, 0, 3, 2, 1],  # 5
    [3, 3, 2, 3, 2, 3, 0, 2, 1],  # 6
    [3, 3, 3, 3, 3, 2, 2, 0, 1],  # 7
    [1, 1, 1, 1, 1, 1, 1, 1, 0],  # 8
]


Dataset

In [ ]:
class DialogueDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512, has_labels=True):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        text = format_dialogue(s["dialogue"])

        encoded = self.tokenizer(
            text,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }

        if self.has_labels:
            label = int(s["label"])
            aux_A, aux_B, aux_C, aux_D = get_aux_labels(label)
            item["label"] = torch.tensor(label, dtype=torch.long)
            item["aux_A"] = torch.tensor(aux_A, dtype=torch.long)
            item["aux_B"] = torch.tensor(aux_B, dtype=torch.long)
            item["aux_C"] = torch.tensor(aux_C, dtype=torch.long)
            item["aux_D"] = torch.tensor(aux_D, dtype=torch.long)
        return item


def collate_batch(batch, pad_token_id=0):
    max_len = max(item["input_ids"].size(0) for item in batch)

    input_ids = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L] = item["input_ids"]
        attention_mask[i, :L] = item["attention_mask"]
    out = {"input_ids": input_ids, "attention_mask": attention_mask}

    if "label" in batch[0]:
        out["label"] = torch.stack([item["label"] for item in batch])
        out["aux_A"] = torch.stack([item["aux_A"] for item in batch])
        out["aux_B"] = torch.stack([item["aux_B"] for item in batch])
        out["aux_C"] = torch.stack([item["aux_C"] for item in batch])
        out["aux_D"] = torch.stack([item["aux_D"] for item in batch])
    return out

Model

In [ ]:
# Encoder transformer (token mean pooling) --> 5 classifier heads (1 main + 4 auxiliary)

class MultiTaskDefenseClassifier(nn.Module):

    def __init__(self, encoder_name, n_classes=9, n_unfrozen_layers=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.head_main = nn.Linear(hidden_size, n_classes)
        self.head_A = nn.Linear(hidden_size, 1)
        self.head_B = nn.Linear(hidden_size, 1)
        self.head_C = nn.Linear(hidden_size, 1)
        self.head_D = nn.Linear(hidden_size, 1)

        self.freeze_encoder(n_unfrozen_layers)

    def freeze_encoder(self, n_unfrozen_layers):
        for p in self.encoder.parameters():
            p.requires_grad = False
        layers = self.encoder.encoder.layer
        n_total = len(layers)
        unfreeze_from = max(0, n_total - n_unfrozen_layers)
        for i in range(unfreeze_from, n_total):
            for p in layers[i].parameters():
                p.requires_grad = True

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)

        return {
            "main": self.head_main(pooled),
            "A": self.head_A(pooled).squeeze(-1),
            "B": self.head_B(pooled).squeeze(-1),
            "C": self.head_C(pooled).squeeze(-1),
            "D": self.head_D(pooled).squeeze(-1),
        }

Loss

In [ ]:
def compute_class_weights_inv(labels, n_classes):
    counts = np.zeros(n_classes, dtype=np.float64)
    for c in labels:
        counts[c] += 1
    counts = np.clip(counts, 1.0, None)
    inv = 1.0 / counts
    weights = inv / inv.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
def distance_penalty(logits, labels, psych_dist):
    # L = mean_b( sum_i probs[b, i] * PSYCH_DIST[label[b], i] )

    probs = F.softmax(logits, dim=-1)
    dist_row = psych_dist[labels]
    per_sample = (probs * dist_row).sum(dim=-1)
    return per_sample.mean()


# binary cross-entropy che ingora i sample coan i target == -1


def masked_bce(logits, targets):
    """
    Binary cross-entropy che ignora i sample con target == -1
    (usato per mascherare la classe 8 sulle teste ausiliarie).
    """
    mask = targets != -1
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)
    return F.binary_cross_entropy_with_logits(logits[mask], targets[mask].float())


def composite_loss(
    outputs,
    labels,
    aux_A,
    aux_B,
    aux_C,
    aux_D,
    class_weights,
    psych_dist,
    aux_weight=0.3,
    dist_weight=0.1,
    label_smoothing=0.1,
):
    # L_main  = CE_weighted_smoothed(logits, label)
    # L_total = L_main + aux_weight * (L_A + L_B + L_C + L_D) + dist_weight * L_distance

    main_logits = outputs["main"]

    L_main_ce = F.cross_entropy(
        main_logits,
        labels,
        weight=class_weights,
        label_smoothing=label_smoothing,
    )
    L_dist = distance_penalty(main_logits, labels, psych_dist)
    L_main = L_main_ce + dist_weight * L_dist

    L_A = masked_bce(outputs["A"], aux_A)
    L_B = masked_bce(outputs["B"], aux_B)
    L_C = masked_bce(outputs["C"], aux_C)
    L_D = masked_bce(outputs["D"], aux_D)

    L_aux = L_A + L_B + L_C + L_D
    L_total = L_main + aux_weight * L_aux

    return L_total


Training

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"]

        outputs = model(input_ids, attention_mask)
        logits = outputs["main"]
        preds = logits.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    macro_f1 = f1_score(
        all_labels, all_preds, average="macro", labels=list(range(9)), zero_division=0
    )
    return macro_f1, np.array(all_preds), np.array(all_labels)


In [ ]:
def train_model(
    train_samples,
    val_samples,
    tokenizer,
    encoder_name,
    n_classes=9,
    n_unfrozen_layers=2,
    aux_weight=0.3,
    dist_weight=0.1,
    label_smoothing=0.1,
    dropout=0.1,
    lr=2e-5,
    weight_decay=1e-4,
    batch_size=16,
    max_epochs=20,
    patience=3,
    max_length=512,
    grad_clip=1.0,
    seed=42,
    save_path=None,
):

    train_ds = DialogueDataset(train_samples, tokenizer, max_length, has_labels=True)
    val_ds = DialogueDataset(val_samples, tokenizer, max_length, has_labels=True)

    pad_id = tokenizer.pad_token_id or 0
    collate = lambda b: collate_batch(b, pad_token_id=pad_id)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate
    )

    model = MultiTaskDefenseClassifier(
        encoder_name=encoder_name,
        n_classes=n_classes,
        n_unfrozen_layers=n_unfrozen_layers,
        dropout=dropout,
    ).to(DEVICE)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)

    train_labels = [int(s["label"]) for s in train_samples]
    class_weights = compute_class_weights_inv(train_labels, n_classes).to(DEVICE)
    psych_dist = torch.tensor(PSYCH_DIST, dtype=torch.float32, device=DEVICE)

    best_val_f1 = -1.0
    best_state = None
    epochs_no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        n_seen = 0
        loss_components = {
            "main_ce": 0.0,
            "dist": 0.0,
            "aux_A": 0.0,
            "aux_B": 0.0,
            "aux_C": 0.0,
            "aux_D": 0.0,
        }

        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            aux_A = batch["aux_A"].to(DEVICE)
            aux_B = batch["aux_B"].to(DEVICE)
            aux_C = batch["aux_C"].to(DEVICE)
            aux_D = batch["aux_D"].to(DEVICE)

            outputs = model(input_ids, attention_mask)
            loss = composite_loss(
                outputs,
                labels,
                aux_A,
                aux_B,
                aux_C,
                aux_D,
                class_weights=class_weights,
                psych_dist=psych_dist,
                aux_weight=aux_weight,
                dist_weight=dist_weight,
                label_smoothing=label_smoothing,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=grad_clip)
            optimizer.step()

            bs = labels.size(0)
            total_loss += loss.item() * bs
            n_seen += bs

        train_loss = total_loss / max(n_seen, 1)

        val_f1, _, _ = evaluate(model, val_loader, DEVICE)
        history.append((epoch, train_loss, val_f1))

        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1 = val_f1
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
            epochs_no_improve = 0
            marker = "★"
            if save_path is not None:
                save_model(
                    best_state,
                    save_path,
                )
        else:
            epochs_no_improve += 1
            marker = ""
        print(f"{epoch:} | loss={train_loss:.4f} | val_F1={val_f1:.4f}{marker}")

        if epochs_no_improve >= patience:
            print(f"Early stopping - (best val F1={best_val_f1:.4f})")
            break
    model.load_state_dict(best_state)
    return model, best_state, best_val_f1, history


In [ ]:
def evaluate_full(
    model, val_samples, tokenizer, batch_size=16, max_length=512, label_names=None
):

    val_ds = DialogueDataset(val_samples, tokenizer, max_length, has_labels=True)
    pad_id = tokenizer.pad_token_id or 0
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_batch(b, pad_token_id=pad_id),
    )

    macro_f1, preds, labels = evaluate(model, val_loader, DEVICE)

    print(f"Macro-F1: {macro_f1:.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            labels,
            preds,
            target_names=label_names,
            labels=list(range(9)),
            digits=4,
            zero_division=0,
        )
    )
    print("Confusion matrix (rows=true, cols=pred, classes 0-8):")
    print(confusion_matrix(labels, preds, labels=list(range(9))))

    return macro_f1, preds, labels


In [ ]:
@torch.no_grad()
def predict_samples(model, samples, tokenizer, batch_size=16, max_length=512):

    has_labels = all("label" in s for s in samples)
    ds = DialogueDataset(samples, tokenizer, max_length, has_labels=has_labels)
    pad_id = tokenizer.pad_token_id or 0
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_batch(b, pad_token_id=pad_id),
    )

    model.eval()
    all_logits = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = model(input_ids, attention_mask)
        all_logits.append(outputs["main"].cpu())
    logits = torch.cat(all_logits, dim=0)
    probs = F.softmax(logits, dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    return preds, probs


In [ ]:
def save_model(model_state, path):
    dirname = os.path.dirname(path)
    torch.save(model_state, path)


def load_model(path, encoder_name, n_classes=9, n_unfrozen_layers=2, dropout=0.1):
    model_state = torch.load(path, map_location=DEVICE)

    model = MultiTaskDefenseClassifier(
        encoder_name=encoder_name,
        n_classes=n_classes,
        n_unfrozen_layers=n_unfrozen_layers,
        dropout=dropout,
    ).to(DEVICE)

    model.load_state_dict(model_state)
    model.eval()

    return model


In [ ]:
ENCODER_NAME = "mental/mental-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
print(f"Loaded {ENCODER_NAME}")
print(f"Model max len: {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loaded mental/mental-bert-base-uncased
Model max len: 512


##EXPERIMENT

In [ ]:
train_samples = load_from_disk(os.path.join(DATA_PATH, "split/train"))
val_samples = load_from_disk(os.path.join(DATA_PATH, "split/val"))
test_samples = load_from_disk(os.path.join(DATA_PATH, "split/test"))

train_labels = [int(s["label"]) for s in train_samples]
val_labels = [int(s["label"]) for s in val_samples]
test_labels = [int(s["label"]) for s in test_samples]

In [ ]:
SAVE_PATH = os.path.join(DATA_PATH, "checkpoints/multitask_best.pt")

model, best_state, best_val_f1, history = train_model(
    train_samples=train_samples,
    val_samples=val_samples,
    tokenizer=tokenizer,
    encoder_name=ENCODER_NAME,
    n_classes=9,
    n_unfrozen_layers=2,
    aux_weight=0.5,
    dist_weight=0.1,
    label_smoothing=0.1,
    dropout=0.1,
    lr=2e-5,
    weight_decay=1e-4,
    batch_size=16,
    max_epochs=20,
    patience=5,
    max_length=512,
    grad_clip=1.0,
    seed=42,
    save_path=SAVE_PATH,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your d

1 | loss=3.5385 | val_F1=0.1281★
2 | loss=3.3059 | val_F1=0.1682★
3 | loss=3.1101 | val_F1=0.2006★
4 | loss=2.8892 | val_F1=0.1937
5 | loss=2.7461 | val_F1=0.2464★
6 | loss=2.6118 | val_F1=0.2496★
7 | loss=2.4988 | val_F1=0.2400
8 | loss=2.4010 | val_F1=0.2521★
9 | loss=2.2828 | val_F1=0.2350
10 | loss=2.1920 | val_F1=0.2666★
11 | loss=2.0895 | val_F1=0.2738★
12 | loss=1.9914 | val_F1=0.2799★
13 | loss=1.8853 | val_F1=0.2540
14 | loss=1.8077 | val_F1=0.2899★
15 | loss=1.7260 | val_F1=0.2894
16 | loss=1.6185 | val_F1=0.2779
17 | loss=1.5405 | val_F1=0.2839
18 | loss=1.4971 | val_F1=0.2526
19 | loss=1.4120 | val_F1=0.2506
Early stopping - (best val F1=0.2899)


In [ ]:
model = load_model(os.path.join(DATA_PATH, "checkpoints/multitask_best.pt"), ENCODER_NAME)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your d

Val

In [ ]:
CLASS_NAMES = ["0", "1", "2", "3", "4", "5", "6", "7", "8"]

val_f1, val_preds, val_true = evaluate_full(
    model,
    val_samples,
    tokenizer,
    batch_size=16,
    max_length=512,
    label_names=CLASS_NAMES,
)


Macro-F1: 0.2899

Classification report:
              precision    recall  f1-score   support

           0     0.4583    0.7333    0.5641        15
           1     0.4286    0.4000    0.4138        15
           2     0.1429    0.1333    0.1379        15
           3     0.2222    0.1333    0.1667        15
           4     0.0909    0.1333    0.1081        15
           5     0.5000    0.2000    0.2857        15
           6     0.3810    0.5333    0.4444        15
           7     0.2381    0.3333    0.2778        15
           8     0.5000    0.1333    0.2105        15

    accuracy                         0.3037       135
   macro avg     0.3291    0.3037    0.2899       135
weighted avg     0.3291    0.3037    0.2899       135

Confusion matrix (rows=true, cols=pred, classes 0-8):
[[11  0  0  0  1  1  0  2  0]
 [ 0  6  3  1  1  0  2  2  0]
 [ 1  0  2  2  4  0  1  4  1]
 [ 3  2  1  2  4  0  2  1  0]
 [ 3  1  2  3  2  1  3  0  0]
 [ 3  1  1  0  4  3  2  1  0]
 [ 0  0  3  1  1  0 

Test

In [ ]:
test_f1, test_preds, test_true = evaluate_full(
    model,
    test_samples,
    tokenizer,
    batch_size=16,
    max_length=512,
    label_names=CLASS_NAMES,
)


Macro-F1: 0.2539

Classification report:
              precision    recall  f1-score   support

           0     0.5455    0.8000    0.6486        75
           1     0.2703    0.3571    0.3077        28
           2     0.1000    0.1875    0.1304        16
           3     0.1176    0.0800    0.0952        25
           4     0.0893    0.2381    0.1299        21
           5     0.0625    0.0769    0.0690        13
           6     0.2838    0.4773    0.3559        44
           7     0.8197    0.4115    0.5479       243
           8     0.0000    0.0000    0.0000         7

    accuracy                         0.4280       472
   macro avg     0.2543    0.2921    0.2539       472
weighted avg     0.5665    0.4280    0.4537       472

Confusion matrix (rows=true, cols=pred, classes 0-8):
[[ 60   2   2   1   0   1   2   6   1]
 [  2  10   4   1   3   1   3   4   0]
 [  0   2   3   1   7   0   2   1   0]
 [  0   3   5   2   7   2   4   2   0]
 [  4   4   4   0   5   0   3   1   0]
 [  3